Importing Necessary Model and Checking GPU

In [ ]:
from google.colab import drive
import os
import pandas as pd
import tensorflow as tf

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Verify GPU is active
gpu_name = tf.test.gpu_device_name()
if gpu_name != '/device:GPU:0':
  print('GPU device not found. Check Notebook Settings!')
else:
  print('Success! Found GPU at: {}'.format(gpu_name))

Mounted at /content/drive
Success! Found GPU at: /device:GPU:0


In [ ]:
# -q stands for 'quiet' mode (hides the list of 5,000+ files being extracted)
# -d specifies the 'destination' directory
!unzip -q /content/sample_data/dataset.zip -d /content/dataset/

Load Data and Map Tri-Risk Labels

In [ ]:
# --- PATHS (Update these to your Drive folder names) ---
BASE_PATH = '/content/sample_data/dataset'
IMG_DIR = os.path.join(BASE_PATH, 'preprocessed_images')
CSV_PATH = os.path.join(BASE_PATH, 'full_df.csv')

df = pd.read_csv(CSV_PATH)

# Mapping ODIR binary columns to your project goals
df['stroke_target'] = df[['H', 'O']].max(axis=1).astype(float)
df['ckd_target'] = df['D'].astype(float)
df['hyp_target'] = df['H'].astype(float)

# Filter for images that exist in your Drive folder
df = df[df['filename'].apply(lambda x: os.path.exists(os.path.join(IMG_DIR, x)))]
print(f"Total validated images for training: {len(df)}")

Total validated images for training: 6392


Build and Train the Multi-Task Model

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Generators (Higher batch size is possible on Colab GPUs)
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = datagen.flow_from_dataframe(
    dataframe=df, directory=IMG_DIR, x_col="filename",
    y_col=["stroke_target", "ckd_target", "hyp_target"],
    target_size=(224, 224), batch_size=32, class_mode="raw", subset="training"
)

# 2. Multi-Task Architecture
base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dense(512, activation='relu')(x)

# One output layer with 3 neurons (Stroke, CKD, Hypertension)
final_output = layers.Dense(3, activation='sigmoid', name='tri_risk_output')(x)
model = models.Model(inputs=base_model.input, outputs=final_output)

# 3. Training on GPU
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_gen, epochs=15)

# 4. Save directly back to your Google Drive
model.save('/content/drive/MyDrive/dataset/retina_tri_risk_model.h5')
print("Model saved to your Google Drive!")

Found 5114 validated image filenames.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 142s 490ms/step - accuracy: 0.3005 - loss: 0.4702
Epoch 2/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 50s 312ms/step - accuracy: 0.2953 - loss: 0.4143
Epoch 3/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 50s 315ms/step - accuracy: 0.3645 - loss: 0.3985
Epoch 4/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 50s 310ms/step - accuracy: 0.3862 - loss: 0.3929
Epoch 5/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 50s 313ms/step - accuracy: 0.3837 - loss: 0.3893
Epoch 6/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 50s 312ms/step - accuracy: 0.3968 - loss: 0.3852
Epoch 7/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 50s 311ms/step - accuracy: 0.4349 - loss: 0.3817
Epoch 8/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 51s 316ms/step - accuracy: 0.4214 - loss: 0.3788
Epoch 9/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 50s 312ms/step - accuracy: 0.4537 - loss: 0.3721
Epoch 10/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 50s 313ms/step - accuracy: 0.4331 - loss: 0.3661
Epoch 11/15
160/160

Model saved to your Google Drive!


In [ ]:
import pandas as pd
df = pd.read_csv('/content/sample_data/dataset/full_df.csv')
print(df.columns)

Index(['ID', 'Patient Age', 'Patient Sex', 'Left-Fundus', 'Right-Fundus',
       'Left-Diagnostic Keywords', 'Right-Diagnostic Keywords', 'N', 'D', 'G',
       'C', 'A', 'H', 'M', 'O', 'filepath', 'labels', 'target', 'filename'],
      dtype='object')


Efficient-Net

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import pandas as pd
import os
from sklearn.utils import class_weight
import numpy as np

# --- 1. Load the Backbone ---
# We use B3 for higher resolution (300x300) to catch vessel narrowing
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(300, 300, 3))
base_model.trainable = True

# --- 2. Build the "Specialist" Model ---
model_hyp = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(1, activation='sigmoid', name='hypertension_output')
])

model_hyp.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# --- 3. Fixed Data Loading (using 'H' and 'filename') ---
CSV_PATH = '/content/sample_data/dataset/full_df.csv'
IMG_DIR = '/content/sample_data/dataset/preprocessed_images/'

df = pd.read_csv(CSV_PATH)

# Fix: Mapping 'H' as the target column
LABEL_COL = 'H'
df[LABEL_COL] = df[LABEL_COL].astype(str)

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = datagen.flow_from_dataframe(
    df,
    directory=IMG_DIR,
    x_col='filename',
    y_col=LABEL_COL,
    target_size=(300, 300),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_gen = datagen.flow_from_dataframe(
    df,
    directory=IMG_DIR,
    x_col='filename',
    y_col=LABEL_COL,
    target_size=(300, 300),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

# Fix the imbalance: Calculating weights for the 'H' column
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weights = dict(enumerate(weights))

# --- 4. Start Training ---
print(f"🚀 Found {len(df)} records. Starting Specialist Training for Hypertension (Column: {LABEL_COL})...")

model_hyp.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weights,
    verbose=1
)

# --- 5. Save the Hybrid Module ---
model_hyp.save('hypertension_specialist.keras')
print("✅ Done! You can now download 'hypertension_specialist.keras' from the file sidebar.")

Found 5114 validated image filenames belonging to 2 classes.
Found 1278 validated image filenames belonging to 2 classes.
🚀 Found 6392 records. Starting Specialist Training for Hypertension (Column: H)...
Epoch 1/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 323s 1s/step - accuracy: 0.6371 - auc: 0.5470 - loss: 0.6912 - val_accuracy: 0.8850 - val_auc: 0.5271 - val_loss: 0.6282
Epoch 2/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 84s 522ms/step - accuracy: 0.6617 - auc: 0.7162 - loss: 0.6377 - val_accuracy: 0.9264 - val_auc: 0.4243 - val_loss: 0.6077
Epoch 3/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 83s 518ms/step - accuracy: 0.6646 - auc: 0.8061 - loss: 0.5884 - val_accuracy: 0.9147 - val_auc: 0.5986 - val_loss: 0.5805
Epoch 4/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 82s 513ms/step - accuracy: 0.7049 - auc: 0.8216 - loss: 0.5630 - val_accuracy: 0.7418 - val_auc: 0.5810 - val_loss: 0.6308
Epoch 5/15
160/160 ━━━━━━━━━━━━━━━━━━━━ 84s 523ms/step - accuracy: 0.7622 - auc: 0.8612 - loss: 0.5193 - val_accuracy: 0.9163 - val_auc: 0.5159 -